#Specific and Load Functions

In [0]:
print("Running Utility Notebook to initialize all functions to use further")

In [0]:
%pip install word2number
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import IntegerType
from word2number import w2n

def word_to_num(value):
    try:
        # If already numeric
        return int(value)
    except:
        try:
            return w2n.word_to_num(value.lower())
        except:
            return None

word_to_num_udf = udf(word_to_num, IntegerType())

In [0]:
from pyspark.sql import functions as func
from pyspark.sql.window import Window
from pyspark.sql.functions import col

def standardize_staff(df):
    return(
        df.withColumn("role",func.lower(col("role")))
        .withColumn("orgin_hub_city",func.initcap(col("hub_location")))
        .withColumn("load_date",func.current_date())
        .withColumn("fullname",func.concat(col("first_name"),func.lit(" "),col("last_name")))
        .withColumn("hub_location",func.initcap(col("hub_location")))
        .drop("first_name","last_name").drop("hub_location")
        .withColumnRenamed("fullname","staff_full_name")
        .withColumn("shipment_id",word_to_num_udf(func.col("shipment_id")).cast("long"))
        .withColumn("age",word_to_num_udf(func.col("age")).cast("long"))
        
    )

def scrub_geotag(df):
    return (
        df
        .withColumn("city_name", func.initcap("city_name"))
        .withColumn("masked_hub_location", func.initcap("country"))
    )

def standardize_shipments(df):
    return (
        df
        .withColumn("domain", func.lit("Logistics"))
        .withColumn("ingestion_timestamp", func.current_date())
        .withColumn("is_expedited", func.lit(False).cast("boolean"))
        .withColumn("shipment_date", func.to_date("shipment_date", "yy-MM-dd"))
        .withColumn("shipment_cost", func.round("shipment_cost", 2))
        .withColumn("shipment_weight_kg", func.col("shipment_weight_kg").cast("double"))
    )

def enrich_shipments(df):
    return (
        df
        .withColumn("route_segment",
            func.concat_ws("-", "source_city", "destination_city"))
        .withColumn("vehicle_identifier",
            func.concat_ws("_", "vehicle_type", "shipment_id"))
        .withColumn("shipment_year", func.year("shipment_date"))
        .withColumn("shipment_month", func.month("shipment_date"))
        .withColumn("is_weekend",
            func.dayofweek("shipment_date").isin([1,7]))
        .withColumn("is_expedited",
            func.col("shipment_status").isin("IN_TRANSIT", "DELIVERED"))
        .withColumn("cost_per_kg",
            func.round(func.col("shipment_cost") / func.col("shipment_weight_kg"), 2))
        .withColumn("tax_amount",
            func.round(func.col("shipment_cost") * 0.18, 2))
        .withColumn("days_since_shipment",
            func.datediff(func.current_date(), "shipment_date"))
        .withColumn("is_high_value",
            func.col("shipment_cost") > 50000)
    )

def split_columns(df):
    return (
        df
        .withColumn("order_prefix", func.substring("order_id", 1, 3))
        .withColumn("order_sequence", func.substring("order_id", 4, 10))
        .withColumn("ship_year", func.year("shipment_date"))
        .withColumn("ship_month", func.month("shipment_date"))
        .withColumn("ship_day", func.dayofmonth("shipment_date"))
        .withColumn("route_lane",
            func.concat_ws("->", "source_city", "destination_city"))
    )

def mask_name(col):
    return func.concat(
        func.substring(col, 1, 2),
        func.lit("****"),
        func.substring(col, -1, 1)
    )
#df1 = spark.read.csv("/Volumes/develop1/shipment/datalake/logistics_source1",header = True,inferSchema=False)
#df2 = spark.read.csv("/Volumes/develop1/shipment/datalake/logistics_source2",header = True,inferSchema=False)
#df3 = df1.unionByName(df2,allowMissingColumns=True)
#display(df3)
#df_display = standardize_staff(df3)
#display(df_display)

def scrub_geotag(df):
    return (
        df
        .withColumn("city_name", func.initcap("city_name"))
        .withColumn("masked_hub_location", func.initcap("country"))
    )



#Generic Functions:

In [0]:
from pyspark.sql.session import SparkSession

def spark_def(app_name = "No AppName Defined"):

    try:
        spark = SparkSession.getActiveSession()
        if spark:
            return spark
    except:
        pass

    return SparkSession.builder.appName(app_name).getOrCreate()


In [0]:

def csvread(spark,path,header = True,inferSchema = False,sep=','):
    csvdf = spark.read.option("header",header)\
                      .option("inferSchema",inferSchema)\
                      .option("sep",sep)\
                      .csv(path)
    return csvdf

def jsonread(spark,path,mline = True,mode = "PERMISSIVE"):
    jsondf = spark.read.option("multiline",mline)\
                       .option("mode",mode)\
                       .json(path)
    return jsondf

In [0]:
def mergeDf(staff1df,staff2df,allowmissingcol=True):
    return staff1df.unionByName(staff2df, allowMissingColumns=allowmissingcol)

In [0]:
def deltawrite(df,path,mode = 'overwrite',format = "delta"):
    return df.write.format(format).mode(mode).save(path)

#def write_table(df, tablename, mode="overwrite"):
    #df.write.mode(mode).format("delta").saveAsTable(tablename)  


def write_table(df,tablename,mode = 'overwrite',format = "delta"):
     df.write.saveAsTable((tablename),mode = "overwrite",format = "delta")
    